In [ ]:
import requests
import pandas as pd
import time


def fetch_steam_reviews(app_id, target_n=5000, language="english", sleep_sec=1.2):
    base_url = f"https://store.steampowered.com/appreviews/{app_id}"
    cursor = "*"
    collected = []
    seen_ids = set()
    page = 1

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0"
    })

    while len(collected) < target_n:
        params = {
            "json": 1,
            "filter": "updated",
            "language": language,      # 英文评论
            "review_type": "all",
            "purchase_type": "all",
            "num_per_page": 100,
            "cursor": cursor
        }

        try:
            response = session.get(base_url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"[ERROR] Page {page} failed: {e}")
            time.sleep(5)
            continue

        reviews = data.get("reviews", [])
        next_cursor = data.get("cursor")

        if not reviews:
            print(f"[STOP] No more reviews on page {page}.")
            break

        added = 0

        for r in reviews:
            review_id = r.get("recommendationid")
            if review_id in seen_ids:
                continue

            seen_ids.add(review_id)
            author = r.get("author", {}) or {}

            collected.append({
                "app_id": app_id,
                "recommendationid": review_id,
                "review_text": r.get("review", ""),
                "voted_up": r.get("voted_up"),
                "timestamp_created": r.get("timestamp_created"),
                "timestamp_updated": r.get("timestamp_updated"),
                "votes_up": r.get("votes_up"),
                "votes_funny": r.get("votes_funny"),
                "weighted_vote_score": r.get("weighted_vote_score"),
                "comment_count": r.get("comment_count"),
                "steam_purchase": r.get("steam_purchase"),
                "received_for_free": r.get("received_for_free"),
                "written_during_early_access": r.get("written_during_early_access"),
                "author_steamid": author.get("steamid"),
                "author_num_games_owned": author.get("num_games_owned"),
                "author_num_reviews": author.get("num_reviews"),
                "author_playtime_forever_min": author.get("playtime_forever"),
                "author_playtime_last_two_weeks_min": author.get("playtime_last_two_weeks"),
                "author_playtime_at_review_min": author.get("playtime_at_review"),
                "author_last_played": author.get("last_played")
            })

            added += 1
            if len(collected) >= target_n:
                break

        print(f"Page {page}: added {added}, total = {len(collected)}/{target_n}")

        if len(collected) >= target_n:
            break

        if not next_cursor or next_cursor == cursor:
            print("[STOP] Cursor did not advance.")
            break

        cursor = next_cursor
        page += 1
        time.sleep(sleep_sec)

    return pd.DataFrame(collected)


def clean_reviews(df, min_hours=10, use_playtime_at_review=True):
    df = df.copy()

    # 清理空文本
    df["review_text"] = df["review_text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = df[df["review_text"] != ""]

    # 选择 playtime 字段
    playtime_col = "author_playtime_at_review_min" if use_playtime_at_review else "author_playtime_forever_min"
    df[playtime_col] = pd.to_numeric(df[playtime_col], errors="coerce")

    # 转小时
    df["playtime_hours"] = df[playtime_col] / 60

    # 筛掉 10h 以下
    df = df[df["playtime_hours"] >= min_hours]

    # 去重
    df = df.drop_duplicates(subset=["recommendationid"]).reset_index(drop=True)

    return df


if __name__ == "__main__":
    APP_ID = 1174180  # Red Dead Redemption 2

    # 1. 抓 5000 条英文评论
    df_raw = fetch_steam_reviews(
        app_id=APP_ID,
        target_n=5000,
        language="english",
        sleep_sec=1.2
    )

    # 保存原始数据
    df_raw.to_csv("rdr2_reviews_raw_english_5000.csv", index=False, encoding="utf-8-sig")

    # 2. 清洗：只保留 10h 以上
    df_clean = clean_reviews(
        df_raw,
        min_hours=10,
        use_playtime_at_review=True
    )

    # 保存清洗后数据
    df_clean.to_csv("rdr2_reviews_clean_english_5000_10h.csv", index=False, encoding="utf-8-sig")

    # 3. 概览
    print("\n===== OVERVIEW =====")
    print("Raw reviews:", len(df_raw))
    print("Clean reviews (>=10h):", len(df_clean))
    print("\nPositive / Negative:")
    print(df_clean["voted_up"].value_counts(dropna=False))

    print("\nSample rows:")
    print(df_clean[[
        "recommendationid",
        "voted_up",
        "playtime_hours",
        "review_text"
    ]].head(10).to_string(index=False))

In [ ]:
print("Raw reviews:", len(df_raw))
print("Clean reviews (>=10h):", len(df_clean))

In [ ]:
df_clean["review_text"] = df_clean["review_text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df_clean["word_count"] = df_clean["review_text"].str.split().str.len()

df_20 = df_clean[df_clean["word_count"] >= 20].copy()
df_40 = df_clean[df_clean["word_count"] >= 40].copy()

print(">=10h:", len(df_clean))
print(">=10h & >=20 words:", len(df_20))
print(">=10h & >=40 words:", len(df_40))

In [ ]:
narrative_dict = {
    "story": ["story", "narrative", "plot"],
    "characters": ["character", "characters", "arthur", "john", "gang", "dutch", "sadie", "hosea"],
    "ending": ["ending", "chapter", "finale"],
    "writing": ["writing", "dialogue"],
    "immersion": ["immersive", "immersion", "world", "journey"],
    "emotion": ["emotional", "emotion", "feel", "felt", "cry", "cried", "attached", "care about"]
}

In [ ]:
import pandas as pd
import re

df = pd.read_csv("rdr2_reviews_clean_english_10h_20words.csv")
text_col = "review_text_clean" if "review_text_clean" in df.columns else "review_text"

df[text_col] = df[text_col].astype(str).str.lower()

narrative_dict = {
    "story": ["story", "narrative", "plot"],
    "characters": ["character", "characters", "arthur", "john", "gang", "dutch", "sadie", "hosea"],
    "ending": ["ending", "chapter", "finale"],
    "writing": ["writing", "dialogue"],
    "immersion": ["immersive", "immersion", "world", "journey"],
    "emotion": ["emotional", "emotion", "feel", "felt", "cry", "cried", "attached", "care about"]
}

# 1. 给每类叙事元素打标签
for category, words in narrative_dict.items():
    pattern = r"\b(?:"
    pattern += "|".join(re.escape(w) for w in words)
    pattern += r")\b"
    df[category] = df[text_col].str.contains(pattern, regex=True, na=False)

# 2. narrative broad: 只要命中任意一类就算
narrative_cols = list(narrative_dict.keys())
df["narrative_broad"] = df[narrative_cols].any(axis=1)

# 3. 正评 / 差评切开
pos_df = df[df["voted_up"] == True].copy()
neg_df = df[df["voted_up"] == False].copy()

print("Total reviews:", len(df))
print("Positive reviews:", len(pos_df))
print("Negative reviews:", len(neg_df))

print("\n=== Q1: Narrative in overall positive reviews ===")
print("Positive reviews mentioning any broad narrative element:",
      pos_df["narrative_broad"].sum())
print("Share of positive reviews mentioning narrative:",
      round(pos_df["narrative_broad"].mean() * 100, 2), "%")

print("\nNarrative category presence within positive reviews:")
for col in narrative_cols:
    count = pos_df[col].sum()
    pct = round(pos_df[col].mean() * 100, 2)
    print(f"{col}: {count} ({pct}%)")

print("\n=== Narrative-related reviews sentiment split ===")
nar_df = df[df["narrative_broad"]].copy()
print(nar_df["voted_up"].value_counts(dropna=False))
print("Positive share among narrative-related reviews:",
      round((nar_df["voted_up"] == True).mean() * 100, 2), "%")

print("\n=== Q2: Broad narrative elements co-occurring with positive experience ===")
# 看正评里每类 narrative element 出现多少
summary = []
for col in narrative_cols:
    summary.append({
        "category": col,
        "positive_count": int(pos_df[col].sum()),
        "positive_pct": round(pos_df[col].mean() * 100, 2)
    })

summary_df = pd.DataFrame(summary).sort_values("positive_count", ascending=False)
print(summary_df)

summary_df.to_csv("rdr2_positive_narrative_summary.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd

# 读 10h+ master
df_master = pd.read_csv("rdr2_reviews_clean_english_5000_10h.csv")

print("Master rows:", len(df_master))

# 清理文本
df_master["review_text"] = df_master["review_text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

# 算词数
df_master["word_count"] = df_master["review_text"].str.split().str.len()

# 重建 20词版
df_20 = df_master[df_master["word_count"] >= 20].copy().reset_index(drop=True)

print("Rebuilt >=20 words rows:", len(df_20))

# 另存为一个新名字，别再覆盖旧文件
df_20.to_csv("rdr2_reviews_clean_english_5000_10h_20words_REBUILT.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd
import re

# 1. 读取主分析集
df = pd.read_csv("rdr2_reviews_clean_english_5000_10h_20words_REBUILT.csv")
text_col = "review_text_clean" if "review_text_clean" in df.columns else "review_text"

df[text_col] = df[text_col].astype(str).str.lower()

print("Total reviews:", len(df))
print("Positive reviews:", (df["voted_up"] == True).sum())
print("Negative reviews:", (df["voted_up"] == False).sum())

# 2. 定义广义叙事元素
narrative_dict = {
    "story": ["story", "narrative", "plot"],
    "characters": ["character", "characters", "arthur", "john", "gang", "dutch", "sadie", "hosea"],
    "ending": ["ending", "chapter", "finale"],
    "writing": ["writing", "dialogue"],
    "immersion": ["immersive", "immersion", "world", "journey"],
    "emotion": ["emotional", "emotion", "feel", "felt", "cry", "cried", "attached", "care about"]
}

# 3. 给每类 narrative element 打标签
for category, words in narrative_dict.items():
    pattern = r"\b(?:"
    pattern += "|".join(re.escape(w) for w in words)
    pattern += r")\b"
    df[category] = df[text_col].str.contains(pattern, regex=True, na=False)

narrative_cols = list(narrative_dict.keys())
df["narrative_broad"] = df[narrative_cols].any(axis=1)

# 4. 切正评 / 差评
pos_df = df[df["voted_up"] == True].copy()
neg_df = df[df["voted_up"] == False].copy()
nar_df = df[df["narrative_broad"]].copy()

# =========================
# Q1: 叙事在整体好评中占据什么位置？
# =========================
print("\n=== Q1: Narrative in overall positive reviews ===")
print("Positive reviews mentioning any broad narrative element:",
      pos_df["narrative_broad"].sum())
print("Share of positive reviews mentioning narrative:",
      round(pos_df["narrative_broad"].mean() * 100, 2), "%")

print("\nNarrative category presence within positive reviews:")
q1_rows = []
for col in narrative_cols:
    count = int(pos_df[col].sum())
    pct = round(pos_df[col].mean() * 100, 2)
    q1_rows.append({
        "category": col,
        "positive_count": count,
        "positive_pct": pct
    })
    print(f"{col}: {count} ({pct}%)")

q1_df = pd.DataFrame(q1_rows).sort_values("positive_count", ascending=False)

print("\n=== Narrative-related reviews sentiment split ===")
print(nar_df["voted_up"].value_counts(dropna=False))
print("Positive share among narrative-related reviews:",
      round((nar_df["voted_up"] == True).mean() * 100, 2), "%")

# =========================
# Q2: 哪些广义叙事元素最常与正向体验一起出现？
# =========================
print("\n=== Q2: Broad narrative elements co-occurring with positive experience ===")
print(q1_df)

# 5. 定义正向体验词
positive_words = [
    "amazing", "masterpiece", "love", "memorable", "immersive",
    "beautiful", "incredible", "best", "favorite", "great", "emotional"
]

for w in positive_words:
    pattern = r"\b" + re.escape(w) + r"\b"
    df[w] = df[text_col].str.contains(pattern, regex=True, na=False)

# 重新切一次，确保有新列
pos_df = df[df["voted_up"] == True].copy()

co_results = []
for nar in narrative_cols:
    for pw in positive_words:
        count = int(((pos_df[nar]) & (pos_df[pw])).sum())
        if count > 0:
            co_results.append({
                "narrative_element": nar,
                "positive_word": pw,
                "count": count
            })

co_df = pd.DataFrame(co_results).sort_values("count", ascending=False)

print("\nTop co-occurrences between narrative elements and positive experience words:")
print(co_df.head(30))

# 6. 保存结果
q1_df.to_csv("rdr2_q1_positive_narrative_presence.csv", index=False, encoding="utf-8-sig")
co_df.to_csv("rdr2_q2_narrative_positive_cooccurrence.csv", index=False, encoding="utf-8-sig")

In [ ]:
mechanical_flaw_terms = [
    "slow", "pacing", "clunky", "boring", "repetitive",
    "controls", "control", "outdated", "tedious",
    "mission design", "missions", "gunplay", "combat"
]

In [ ]:
rdr2_flaw_terms = [
    "slow", "pacing", "clunky", "boring", "repetitive",
    "controls", "control", "outdated", "tedious",
    "mission design", "missions", "gunplay", "combat",
    "linear", "restrictive", "hand holding", "hand-holding",
    "unskippable", "too much riding", "riding", "waiting",
    "animations", "realism", "sluggish"
]

In [ ]:
import pandas as pd
import re

# 1. 读数据
df = pd.read_csv("rdr2_reviews_clean_english_5000_10h_20words_REBUILT.csv")
text_col = "review_text_clean" if "review_text_clean" in df.columns else "review_text"
df[text_col] = df[text_col].astype(str).str.lower()

# 2. broad narrative categories
narrative_dict = {
    "story": ["story", "narrative", "plot"],
    "characters": ["character", "characters", "arthur", "john", "gang", "dutch", "sadie", "hosea"],
    "ending": ["ending", "chapter", "finale"],
    "writing": ["writing", "dialogue"],
    "immersion": ["immersive", "immersion", "world", "journey"],
    "emotion": ["emotional", "emotion", "feel", "felt", "cry", "cried", "attached", "care about"]
}

for category, words in narrative_dict.items():
    pattern = r"\b(?:" + "|".join(re.escape(w) for w in words) + r")\b"
    df[category] = df[text_col].str.contains(pattern, regex=True, na=False)

narrative_cols = list(narrative_dict.keys())
df["narrative_broad"] = df[narrative_cols].any(axis=1)

# 3. 你的 RDR2 flaw terms
rdr2_flaw_terms = [
    "slow", "pacing", "clunky", "boring", "repetitive",
    "controls", "control", "outdated", "tedious",
    "mission design", "missions", "gunplay", "combat",
    "linear", "restrictive", "hand holding", "hand-holding",
    "unskippable", "too much riding", "riding", "waiting",
    "animations", "realism", "sluggish"
]

flaw_pattern = r"\b(?:" + "|".join(re.escape(x) for x in rdr2_flaw_terms) + r")\b"
df["mechanical_flaw"] = df[text_col].str.contains(flaw_pattern, regex=True, na=False)

# 4. 基本统计
print("Total reviews:", len(df))
print("Mechanical flaw comments:", int(df["mechanical_flaw"].sum()))

flaw_df = df[df["mechanical_flaw"]].copy()
flaw_pos = flaw_df[flaw_df["voted_up"] == True].copy()
flaw_neg = flaw_df[flaw_df["voted_up"] == False].copy()

print("\n=== Mechanical flaw comments ===")
print("Positive flaw comments:", len(flaw_pos))
print("Negative flaw comments:", len(flaw_neg))

print("\nNarrative broad share in positive flaw comments:")
print(round(flaw_pos["narrative_broad"].mean() * 100, 2), "%")

print("Narrative broad share in negative flaw comments:")
print(round(flaw_neg["narrative_broad"].mean() * 100, 2), "%")

# 5. narrative categories 对比
rows = []
for col in narrative_cols:
    rows.append({
        "category": col,
        "positive_flaw_count": int(flaw_pos[col].sum()),
        "positive_flaw_pct": round(flaw_pos[col].mean() * 100, 2) if len(flaw_pos) > 0 else 0,
        "negative_flaw_count": int(flaw_neg[col].sum()),
        "negative_flaw_pct": round(flaw_neg[col].mean() * 100, 2) if len(flaw_neg) > 0 else 0
    })

comp_df = pd.DataFrame(rows).sort_values("positive_flaw_pct", ascending=False)

print("\n=== Narrative categories in flaw comments ===")
print(comp_df)

# 6. 导出
comp_df.to_csv("rdr2_narrative_compensation_flaw_comparison_v2.csv", index=False, encoding="utf-8-sig")

# 7. 把“提到缺陷但仍给好评 / 差评”的原文分别导出
flaw_pos.to_csv("rdr2_positive_flaw_comments_v2.csv", index=False, encoding="utf-8-sig")
flaw_neg.to_csv("rdr2_negative_flaw_comments_v2.csv", index=False, encoding="utf-8-sig")

In [ ]:
neg_flaw_only = flaw_neg.copy()

neg_flaw_only.to_csv("rdr2_negative_flaw_comments.csv", index=False, encoding="utf-8-sig")

print("Negative flaw comments:", len(neg_flaw_only))
print(neg_flaw_only[["review_text"]].head(20).to_string(index=False))